#### Travel planning 

In [8]:

query = input("Enter your query: ")

# context to get from database
context = {
    
}

print(f"Entered query: {query}")

Entered query: plan a trip from delhi to los angeles from 27th jan 2026 for 5 days


In [9]:
#llama model to extract travel details (our fine tuned model will be used here)
import requests
import json
from typing import Optional, Dict
from datetime import datetime

# Ollama API configuration
OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL_NAME = "llama3.2:latest"  

EXTRACTION_PROMPT = """You are a travel information extraction assistant. Extract ONLY the following information from the user's query and return it as JSON.

Required JSON format (use null for missing values):
{
  "from_city": "CityName",
  "to_city": "CityName", 
  "check_in": "YYYY-MM-DD",
  "check_out": "YYYY-MM-DD",
  "num_nights": number,
  "num_adults": number
}

Rules:
- If "from" or "starting from" appears, extract from_city
- If "to" or "going to" appears, extract to_city
- Convert date mentions like "27 january 2026" to "2026-01-27" format
- If "X days" or "X nights" appears, extract as num_nights
- If "X people" or "X adults" appears, extract as num_adults
- Use null for any missing information
- Return ONLY the JSON object, nothing else

Examples:

Query: "Trip to Goa from Dec 15 to Dec 20"
{"from_city": null, "to_city": "Goa", "check_in": "2025-12-15", "check_out": "2025-12-20", "num_nights": 5, "num_adults": null}

Query: "Plan a trip from Mumbai to Delhi for 3 nights"  
{"from_city": "Mumbai", "to_city": "Delhi", "check_in": null, "check_out": null, "num_nights": 3, "num_adults": null}

Query: "Book Bangalore to Goa Jan 26 for 2 people"
{"from_city": "Bangalore", "to_city": "Goa", "check_in": "2026-01-26", "check_out": null, "num_nights": null, "num_adults": 2}

Now extract from this query:"""

def extract_details(query: str) -> Dict[str, Optional[str]]:
    """Extract travel details using Ollama API with JSON mode."""
    
    payload = {
        "model": MODEL_NAME,
        "prompt": f"{EXTRACTION_PROMPT}\n\nQuery: {query}\nOutput:",
        "format": "json",
        "stream": False,
        "options": {
            "temperature": 0.0,
            "num_predict": 256
        }
    }
    
    print("=" * 50)
    print("DEBUG: Starting extraction...")
    print(f"DEBUG: Query = {query}")
    print(f"DEBUG: Ollama URL = {OLLAMA_URL}")
    print(f"DEBUG: Model = {MODEL_NAME}")
    
    try:
        print("DEBUG: Sending request to Ollama...")
        response = requests.post(OLLAMA_URL, json=payload, timeout=30)
        
        print(f"DEBUG: Response status code = {response.status_code}")
        print(f"DEBUG: Response headers = {response.headers}")
        
        response.raise_for_status()
        
        result = response.json()
        print(f"DEBUG: Full result = {result}")
        
        text = result["response"].strip()
        print(f"DEBUG: Raw LLaMA response = '{text}'")
        
        data = json.loads(text)
        print(f"DEBUG: Parsed JSON = {data}")
        
        # Normalize and validate
        extracted = {
            "from_city": data.get("from_city", "").strip().title() if data.get("from_city") else None,
            "to_city": data.get("to_city", "").strip().title() if data.get("to_city") else None,
            "check_in": validate_date(data.get("check_in")),
            "check_out": validate_date(data.get("check_out")),
            "num_nights": data.get("num_nights") if isinstance(data.get("num_nights"), int) else None,
            "num_adults": data.get("num_adults") if isinstance(data.get("num_adults"), int) else 1
        }
        
        print(f"DEBUG: Final extracted = {extracted}")
        print("=" * 50)
        return extracted
    
    except requests.RequestException as e:
        print(f" Ollama API error: {e}")
        print(f" Error type: {type(e).__name__}")
        print(" Make sure Ollama is running: ollama serve")
        return _empty_extraction()
    
    except json.JSONDecodeError as e:
        print(f" JSON parse error: {e}")
        print(f" Raw text that failed to parse: '{text}'")
        return _empty_extraction()
    
    except Exception as e:
        print(f" Unexpected error: {e}")
        print(f" Error type: {type(e).__name__}")
        import traceback
        traceback.print_exc()
        return _empty_extraction()

def validate_date(date_str: Optional[str]) -> Optional[str]:
    """Validate date format YYYY-MM-DD"""
    if not date_str:
        return None
    try:
        datetime.strptime(date_str, '%Y-%m-%d')
        return date_str
    except ValueError:
        return None

def _empty_extraction() -> Dict[str, Optional[str]]:
    """Return empty extraction result"""
    return {
        "from_city": None,
        "to_city": None,
        "check_in": None,
        "check_out": None,
        "num_nights": None,
        "num_adults": 1
    }

In [10]:
def route_flight_api(llama_entities):
    from_city = llama_entities.get("from_city")
    to_city = llama_entities.get("to_city")
    actual_source = None
    # If no destination at all → no flight
    if not to_city:
        return "SKIP_FLIGHT", None, None

    # Priority 1: If user explicitly gave 'from'
    if from_city:
        actual_source = from_city
    else:
        if context.get("default_loc"):
            actual_source = context.get("default_loc")

    # If still no source → skip for now
    if not actual_source:
        return "SKIP_FLIGHT",None, None

    # If source and destination are same → no flight
    if actual_source.lower() == to_city.lower():
        return "SKIP_FLIGHT", actual_source, to_city

    # Otherwise → call flight API
    return "CALL_FLIGHT_API", actual_source, to_city


In [11]:
from amadeus import Client, ResponseError

amadeus = Client(
    client_id="GL4lMSLONHWXs0kroqnYabMGjaqzXAHR",
    client_secret="CA25nHIoPpmb1ks6"
)

def city_name_to_code(amadeus, city_name):
    locations = amadeus.reference_data.locations.get(
        keyword=city_name,
        subType="CITY"
    )
    return locations.data[0]["iataCode"]

def flight_search(amadeus, origin_code, destination_code, departure_date, adults=1, max_results=5):
    response = amadeus.shopping.flight_offers_search.get(
        originLocationCode=origin_code,
        destinationLocationCode=destination_code,
        departureDate=departure_date,
        adults=adults,
        max=max_results
    )
    return response.data




In [12]:
def route_hotel_api(llama_entities, flight_search_results):
    """
    Decide whether to search hotels based on extracted entities and flight results.
    Returns: (decision: str, params: dict)
    """
    to_city = llama_entities.get("to_city")
    check_in = llama_entities.get("check_in")
    check_out = llama_entities.get("check_out")
    num_nights = llama_entities.get("num_nights")
    num_adults = llama_entities.get("num_adults", 1)
    
    # No destination → skip
    if not to_city:
        return "SKIP_HOTEL", None
    
    # If we have check_in and check_out → search with pricing
    if check_in and check_out:
        return "CALL_HOTEL_API", {
            "city": to_city,
            "check_in": check_in,
            "check_out": check_out,
            "num_adults": num_adults
        }
    
    # If we have check_in and num_nights → calculate check_out
    if check_in and num_nights:
        check_out_date = (datetime.strptime(check_in, '%Y-%m-%d') + 
                         __import__('datetime').timedelta(days=num_nights)).strftime('%Y-%m-%d')
        return "CALL_HOTEL_API", {
            "city": to_city,
            "check_in": check_in,
            "check_out": check_out_date,
            "num_adults": num_adults
        }
    
    # If we have check_in only → need checkout
    if check_in:
        return "ASK_DATES", {"city": to_city}
    
    # No dates at all → skip
    return "SKIP_HOTEL", None

def get_hotel_offers(amadeus, city_code, check_in, check_out, num_adults=1, max_results=5):
    """
    Two-step hotel search:
    1. Get hotel IDs by city
    2. Get offers for those hotels
    """
    try:
        # STEP 1: Get hotels in the city
        print(f"DEBUG: Step 1 - Getting hotels in {city_code}...")
        
        hotels_response = amadeus.reference_data.locations.hotels.by_city.get(
            cityCode=city_code
        )
        
        if not hotels_response.data:
            print(f" No hotels found in {city_code}")
            return {
                "status": "OK",
                "hotels": []
            }
        
        # Get hotel IDs (limit to first 10-20 for performance)
        hotel_ids = [hotel['hotelId'] for hotel in hotels_response.data[:20]]
        print(f"DEBUG: Found {len(hotel_ids)} hotels in {city_code}")
        
        # STEP 2: Get offers for these hotels
        print(f"DEBUG: Step 2 - Getting offers for hotels...")
        
        offers_response = amadeus.shopping.hotel_offers_search.get(
            hotelIds=hotel_ids,
            checkInDate=check_in,
            checkOutDate=check_out,
            adults=num_adults,
            roomQuantity=1
        )
        
        print(f"DEBUG: Got offers for {len(offers_response.data)} hotels")
        
        data = offers_response.data[:max_results]

        return {
            "status": "OK",
            "hotels": parse_hotel_offers(data)
        }

    except ResponseError as error:
        print(f" Amadeus API Error: {error}")
        if hasattr(error, 'response'):
            print(f"   Status: {error.response.status_code}")
            print(f"   Details: {error.response.body}")
        return {
            "status": "ERROR",
            "message": str(error),
            "hotels": []
        }
    except Exception as error:
        print(f"Unexpected error: {error}")
        import traceback
        traceback.print_exc()
        return {
            "status": "ERROR",
            "message": str(error),
            "hotels": []
        }
    
def parse_hotel_offers(data):
    results = []

    for item in data:
        hotel = item.get("hotel", {})
        offers = item.get("offers", [])

        if not offers:
            continue

        offer = offers[0]   # take cheapest offer

        results.append({
            "hotel_id": hotel.get("hotelId"),
            "name": hotel.get("name"),
            "city": hotel.get("cityCode"),
            "price": offer.get("price", {}).get("total"),
            "currency": offer.get("price", {}).get("currency"),
            "room_type": offer.get("room", {}).get("typeEstimated", {}).get("category"),
            "description": offer.get("room", {}).get("description", {}).get("text"),
            "cancellation": offer.get("policies", {}).get("cancellation", {}).get("description", {}).get("text")
        })

    return results



In [13]:
import requests
import json
from typing import List, Dict

OVERPASS_URL = "https://overpass-api.de/api/interpreter"

def get_attractions_from_osm(city: str, num_days: int, max_attractions: int = 20) -> List[Dict]:
    """Fetch top attractions from OpenStreetMap with multiple fallback strategies"""
    
    print(f"🔍 Fetching attractions in {city}...")
    
    # Helper function to get city bounding box from Nominatim
    def get_city_bbox(city_name):
        """Get bounding box from Nominatim geocoding"""
        nominatim_url = "https://nominatim.openstreetmap.org/search"
        try:
            print(f"   📍 Getting coordinates for {city_name} from Nominatim...")
            response = requests.get(
                nominatim_url,
                params={
                    "q": city_name,
                    "format": "json",
                    "limit": 1
                },
                headers={"User-Agent": "TravelPlanner/1.0"},
                timeout=10
            )
            if response.status_code == 200:
                data = response.json()
                if data:
                    bbox = data[0].get("boundingbox")  # [min_lat, max_lat, min_lon, max_lon]
                    display_name = data[0].get("display_name", "")
                    print(f"   ✅ Found: {display_name}")
                    print(f"   📦 Bounding box: {bbox}")
                    if bbox:
                        return bbox
        except Exception as e:
            print(f"   Nominatim error: {e}")
        return None
    
    attractions = []
    
    # STRATEGY 1: Try geocodeArea first (works for well-known cities)
    print(f"\n  Strategy 1: geocodeArea for '{city}'")
    query_geocode = f"""
    [out:json][timeout:30];
    {{geocodeArea:{city}}}->.searchArea;
    (
      nwr["tourism"~"museum|attraction|viewpoint"](area.searchArea);
      nwr["amenity"~"aquarium|zoo|arts_centre|theatre"](area.searchArea);
      nwr["leisure"="park"]["name"](area.searchArea);
      nwr["man_made"="tower"](area.searchArea);
      nwr["historic"~"monument|memorial|fort|palace"](area.searchArea);
      nwr["building"="cathedral"](area.searchArea);
    );
    out center tags 100;
    """
    
    try:
        response = requests.post(
            OVERPASS_URL, 
            data={"data": query_geocode}, 
            timeout=60,
            headers={"User-Agent": "TravelPlanner/1.0"}
        )
        
        if response.status_code == 200:
            data = response.json()
            elements = data.get("elements", [])
            print(f"  Found {len(elements)} elements")
            
            if elements:
                attractions = parse_attractions(elements)
                if attractions:
                    print(f"  Success with geocodeArea! Got {len(attractions)} attractions")
        else:
            print(f"   geocodeArea failed with status {response.status_code}")
            
    except Exception as e:
        print(f"  geocodeArea error: {e}")
    
    # STRATEGY 2: If geocodeArea fails, use Nominatim to get bbox and search within it
    if not attractions:
        print(f"\n  Strategy 2: Nominatim + Bounding Box Search")
        bbox = get_city_bbox(city)
        
        if bbox:
            min_lat, max_lat, min_lon, max_lon = bbox
            
            query_with_bbox = f"""
            [out:json][timeout:30];
            (
              nwr["tourism"~"museum|attraction|viewpoint"]({min_lat},{min_lon},{max_lat},{max_lon});
              nwr["amenity"~"aquarium|zoo|arts_centre|theatre"]({min_lat},{min_lon},{max_lat},{max_lon});
              nwr["leisure"="park"]["name"]({min_lat},{min_lon},{max_lat},{max_lon});
              nwr["man_made"="tower"]({min_lat},{min_lon},{max_lat},{max_lon});
              nwr["historic"~"monument|memorial|fort|palace"]({min_lat},{min_lon},{max_lat},{max_lon});
              nwr["building"="cathedral"]({min_lat},{min_lon},{max_lat},{max_lon});
            );
            out center tags 100;
            """
            
            try:
                print(f" Searching within bounding box...")
                response = requests.post(
                    OVERPASS_URL,
                    data={"data": query_with_bbox},
                    timeout=60,
                    headers={"User-Agent": "TravelPlanner/1.0"}
                )
                
                if response.status_code == 200:
                    data = response.json()
                    elements = data.get("elements", [])
                    print(f" Bbox search found {len(elements)} elements")
                    attractions = parse_attractions(elements)
                    if attractions:
                        print(f"  Success with bbox search! Got {len(attractions)} attractions")
                else:
                    print(f"  Bbox search failed with status {response.status_code}")
                        
            except Exception as e:
                print(f"    Bbox search error: {e}")
        else:
            print(f"     Could not get bounding box for {city}")
    
    if not attractions:
        print(f"\n No attractions found for {city} after all strategies")
        print(f" This could mean:")
        print(f"   - The city name needs to be more specific (e.g., 'New York City' instead of 'New York')")
        print(f"   - OSM has limited data for this location")
        print(f"   - Try adding country name: '{city}, USA' or '{city}, France'")
        return []
    
    # Sort by score and return top attractions
    attractions.sort(key=lambda x: x["score"], reverse=True)
    top_attractions = attractions[:max_attractions]
    
    print(f"\n Final result: {len(top_attractions)} top attractions")
    
    # Debug: print top 5
    if top_attractions:
        print(f"\n Top {min(5, len(top_attractions))} attractions:")
        for i, attr in enumerate(top_attractions[:5], 1):
            print(f"   {i}. {attr['name']} ({attr['category']}) - Score: {attr['score']}")
    
    return top_attractions


def parse_attractions(elements: list) -> list:
    """Extract and score attractions from OSM elements"""
    attractions = []
    seen_coords = set()
    
    for element in elements:
        tags = element.get("tags", {})
        name = tags.get("name")
        
        if not name or len(name) < 3:
            continue
        
        # Get coordinates
        lat = element.get("lat") or element.get("center", {}).get("lat")
        lon = element.get("lon") or element.get("center", {}).get("lon")
        
        if not lat or not lon:
            continue
        
        # Deduplication
        coord_key = (round(lat, 3), round(lon, 3))
        if coord_key in seen_coords:
            continue
        seen_coords.add(coord_key)
        
        # Score and categorize
        score = score_attraction(tags)
        category = categorize_attraction(tags)
        
        attractions.append({
            "name": name,
            "category": category,
            "score": score,
            "lat": lat,
            "lon": lon,
            "tags": tags
        })
    
    return attractions


def score_attraction(tags: Dict) -> int:
    """Score attraction based on importance indicators"""
    score = 0
    
    # Wikipedia/Wikidata = famous
    if "wikidata" in tags: score += 15
    if "wikipedia" in tags: score += 10
    
    # Type-based scoring
    tourism = tags.get("tourism", "")
    historic = tags.get("historic", "")
    amenity = tags.get("amenity", "")
    
    if tourism == "museum": score += 12
    if tourism == "attraction": score += 10
    if historic in ["fort", "palace", "monument"]: score += 11
    if amenity in ["aquarium", "zoo"]: score += 9
    if tags.get("man_made") == "tower": score += 10
    if tags.get("leisure") == "park": score += 7
    if tourism == "viewpoint": score += 6
    
    # Popularity indicators
    if "website" in tags: score += 3
    if "opening_hours" in tags: score += 2
    
    return score


def categorize_attraction(tags: Dict) -> str:
    """Categorize attraction for better descriptions"""
    if tags.get("tourism") == "museum": return "Museum"
    if tags.get("historic"): return "Historical Site"
    if tags.get("leisure") == "park": return "Park"
    if tags.get("amenity") in ["aquarium", "zoo"]: return "Family Attraction"
    if tags.get("tourism") == "viewpoint": return "Viewpoint"
    if tags.get("man_made") == "tower": return "Landmark"
    return "Tourist Attraction"


# ============================================================================
# STEP 2: Use LLaMA to Generate Smart Itinerary
# ============================================================================

def generate_itinerary_with_llama(
    city: str, 
    attractions: List[Dict], 
    num_days: int,
    check_in: str,
    num_adults: int = 1,
    flight_info: str = None,
    hotel_info: str = None
) -> str:
    """Use LLaMA to generate a personalized day-by-day itinerary"""
    
    if not attractions:
        return "No attractions available to create itinerary."
    
    # Format attractions for LLaMA
    attractions_text = "\n".join([
        f"- {a['name']} ({a['category']}) - Score: {a['score']}"
        for a in attractions[:15]  # Limit to top 15
    ])
    
    # Create context for LLaMA
    context_parts = [
        f"Destination: {city}",
        f"Trip Duration: {num_days} days",
        f"Check-in Date: {check_in}",
        f"Number of Travelers: {num_adults}"
    ]
    
    if flight_info:
        context_parts.append(f"Flight Info: {flight_info}")
    if hotel_info:
        context_parts.append(f"Hotel Info: {hotel_info}")
    
    context = "\n".join(context_parts)
    
    # LLaMA prompt
    prompt = f"""You are an expert travel planner. Create a detailed, realistic {num_days}-day itinerary for {city}.

TRIP DETAILS:
{context}

AVAILABLE ATTRACTIONS (sorted by popularity):
{attractions_text}

INSTRUCTIONS:
1. Create a day-by-day plan (Day 1, Day 2, etc.)
2. Include 2-4 attractions per day (don't overcrowd the schedule)
3. Add realistic timings (e.g., 9:00 AM, 2:00 PM)
4. Include meal breaks (breakfast, lunch, dinner)
5. Consider travel time between attractions
6. Group nearby attractions together
7. Mix different types of attractions (museums, parks, landmarks)
8. Leave some free time for relaxation
9. Make it practical and enjoyable

FORMAT:
Use clear headings like "Day 1:", "Day 2:", etc.
Include times, activities, and brief descriptions.

Generate the itinerary now:"""
    
    print(f"\n Generating personalized itinerary with LLaMA...")
    
    payload = {
        "model": MODEL_NAME,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0.7,  # Higher for creativity
            "num_predict": 1500   # Longer response for detailed itinerary
        }
    }
    
    try:
        response = requests.post(OLLAMA_URL, json=payload, timeout=60)
        response.raise_for_status()
        
        result = response.json()
        itinerary = result["response"].strip()
        
        return itinerary
        
    except Exception as e:
        print(f" Error generating itinerary: {e}")
        return f"Error: Could not generate itinerary. {str(e)}"


def display_itinerary(itinerary: str, city: str, num_days: int):
    """Display the generated itinerary in a nice format"""
    
    print(f"  {num_days}-DAY PERSONALIZED ITINERARY FOR {city.upper()}")
    
    print(itinerary)
    

    print(" Enjoy your trip! Have an amazing time! ✨")
 


# ============================================================================
# STEP 3: Main Integration Function
# ============================================================================

def plan_trip_with_llama(
    llama_entities: Dict, 
    flight_results: List = None, 
    hotel_results: Dict = None
):
    """
    Complete trip planning using LLaMA for intelligent itinerary generation
    """
    
    to_city = llama_entities.get("to_city")
    check_in = llama_entities.get("check_in")
    num_nights = llama_entities.get("num_nights")
    num_adults = llama_entities.get("num_adults", 1)
    
    if not to_city or not check_in or not num_nights:
        print(" Missing trip details for itinerary planning")
        return None
    
    # Step 1: Get attractions from OpenStreetMap
    attractions = get_attractions_from_osm(
        city=to_city, 
        num_days=num_nights,
        max_attractions=20
    )
    
    if not attractions:
        print(f" No attractions found for {to_city}. Cannot create itinerary.")
        return None
    
    # Step 2: Format flight and hotel info for context
    flight_info = None
    if flight_results and len(flight_results) > 0:
        best_flight = flight_results[0]
        price = best_flight['price']['total']
        currency = best_flight['price']['currency']
        duration = best_flight['itineraries'][0]['duration'].replace('PT', '').replace('H', 'h ').replace('M', 'm')
        flight_info = f"{currency} {price}, Duration: {duration}"
    
    hotel_info = None
    if hotel_results and hotel_results.get("hotels"):
        best_hotel = hotel_results["hotels"][0]
        hotel_info = f"{best_hotel['name']} - {best_hotel['currency']} {best_hotel['price']}/night"
    
    # Step 3: Generate itinerary with LLaMA
    itinerary = generate_itinerary_with_llama(
        city=to_city,
        attractions=attractions,
        num_days=num_nights,
        check_in=check_in,
        num_adults=num_adults,
        flight_info=flight_info,
        hotel_info=hotel_info
    )
    
    # Step 4: Display the itinerary
    display_itinerary(itinerary, to_city, num_nights)
    
    return {
        "attractions": attractions,
        "itinerary": itinerary
    }


In [14]:
llama_entities = extract_details(query)
print(f"Extracted details: {llama_entities}")

##Flight first
decision, src, dest= route_flight_api(llama_entities)
print(f"Routing decision: {decision}, Source: {src}, Destination: {dest}")

flight_search_results = None

if decision == "CALL_FLIGHT_API":
    from_code = city_name_to_code(amadeus, src)
    to_code =  city_name_to_code(amadeus, dest)
    flight_search_results = flight_search(amadeus, from_code, to_code, "2026-01-26")
    
    if flight_search_results:
            print(f"\n Found {len(flight_search_results)} flight(s)\n")

            # Display top 5 results
            for i, offer in enumerate(flight_search_results[:5], 1):
                price = offer['price']['total']
                currency = offer['price']['currency']
                segments = offer['itineraries'][0]['segments']
                
                # Format times nicely
                dep_time = segments[0]['departure']['at'].split('T')[1][:5]
                arr_time = segments[-1]['arrival']['at'].split('T')[1][:5]
                
                # Parse duration (PT format like PT2H30M)
                duration = offer['itineraries'][0]['duration']
                duration_clean = duration.replace('PT', '').replace('H', 'h ').replace('M', 'm')
                
                print(f"{i}. {currency} {price} | {dep_time} → {arr_time} | {duration_clean} | {len(segments)} stop(s)")
else:
    print("→ Skipping Flight API")

# now hotel
hotel_results = None

hotel_decision, hotel_params = route_hotel_api(llama_entities, flight_search_results)
print(f"\n Hotel routing decision: {hotel_decision}, Parameters: {hotel_params}")
if hotel_decision == "CALL_HOTEL_API":
    city_code = to_code
    check_in = hotel_params["check_in"]
    check_out = hotel_params["check_out"]
    num_adults = hotel_params.get("num_adults", 1)

    hotel_results = get_hotel_offers(
        amadeus=amadeus,
        city_code=city_code,
        check_in=check_in,
        check_out=check_out,
        num_adults=num_adults,
        max_results=5
    )

    if hotel_results["status"] == "OK" and hotel_results["hotels"]:
        print(f"\n Found {len(hotel_results['hotels'])} hotel option(s)\n")

        for i, hotel in enumerate(hotel_results["hotels"], 1):
            print(
                f"{i}. {hotel['name']} | "
                f"{hotel['currency']} {hotel['price']} per night | "
                f"Rating: {hotel.get('rating', 'N/A')} | "
                f"Room: {hotel.get('room_type', 'N/A')}"
            )
    else:
        print("→ No hotel offers found.")


trip_plan = plan_trip_with_llama(
    llama_entities=llama_entities,
    flight_results=flight_search_results,
    hotel_results=hotel_results 
)

        

DEBUG: Starting extraction...
DEBUG: Query = plan a trip from delhi to los angeles from 27th jan 2026 for 5 days
DEBUG: Ollama URL = http://localhost:11434/api/generate
DEBUG: Model = llama3.2:latest
DEBUG: Sending request to Ollama...
DEBUG: Response status code = 200
DEBUG: Response headers = {'Content-Type': 'application/json; charset=utf-8', 'Date': 'Mon, 26 Jan 2026 05:42:03 GMT', 'Transfer-Encoding': 'chunked'}
DEBUG: Full result = {'model': 'llama3.2:latest', 'created_at': '2026-01-26T05:42:03.3907317Z', 'response': '{"from_city": "Delhi", "to_city": "Los Angeles", "check_in": "2026-01-27", "check_out": null, "num_nights": 5, "num_adults": null}', 'done': True, 'done_reason': 'stop', 'context': [128006, 9125, 128007, 271, 38766, 1303, 33025, 2696, 25, 6790, 220, 2366, 18, 271, 128009, 128006, 882, 128007, 271, 2675, 527, 264, 5944, 2038, 33289, 18328, 13, 23673, 27785, 279, 2768, 2038, 505, 279, 1217, 596, 3319, 323, 471, 433, 439, 4823, 382, 8327, 4823, 3645, 320, 817, 854, 369